In [4]:
# 1. Mount Drive (if not already mounted)
from google.colab import drive
drive.mount('/content/drive')

# 2. Unzip your images
!unzip -n -q "/content/drive/MyDrive/DATN/renamed_images.zip" -d "/content/"
print("✅ Images unzipped.")

# 3. Use the stable PIP package instead of the Git link
!pip install segment-anything
print("✅ Segment Anything installed.")

# 4. Download weights directly to Colab (Fastest way)
import os
if not os.path.exists("sam_vit_h_4b8939.pth"):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
    print("✅ Weights downloaded.")
else:
    print("✅ Weights already exist.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Images unzipped.
✅ Segment Anything installed.
✅ Weights already exist.


In [5]:
print("Folders in /content/:")
print(os.listdir("/content"))

Folders in /content/:
['.config', 'images', 'sam_vit_h_4b8939.pth', 'drive', 'sample_data']


In [6]:
# ==========================================
# CELL 1: IMPORTS + FILTERED METADATA SETUP
# ==========================================

import pandas as pd
import os
import cv2
import torch
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# --- CONFIGURATION ---
CSV_PATH = "/content/drive/MyDrive/DATN/linked_egg_metadata.csv"
INPUT_DIR = "/content/images"
SAM_CHECKPOINT = "/content/sam_vit_h_4b8939.pth"

# --- LOAD METADATA ---
print("="*60)
print("[1/5] LOADING METADATA...")
df = pd.read_csv(CSV_PATH)

print(f"Total rows in CSV: {len(df)}")

# --- FILTER TO ONLY FILES IN CURRENT SPLIT ---
print("[1.5/5] FILTERING TO CURRENT INPUT FOLDER...")

# Get all filenames in INPUT_DIR
available_images = set(os.listdir(INPUT_DIR))

# Keep only rows where image exists in this split
df = df[df['image_path'].isin(available_images)].reset_index(drop=True)

print(f"Images found in '{INPUT_DIR}': {len(df)}")

# --- OPTIONAL: sanity check ---
missing = df[~df['image_path'].isin(available_images)]
if len(missing) > 0:
    print(f"Warning: {len(missing)} rows still mismatched (unexpected)")

print(df.head())
print("="*60)

[1/5] LOADING METADATA...
Total rows in CSV: 4633
[1.5/5] FILTERING TO CURRENT INPUT FOLDER...
Images found in '/content/images': 4633
   dot  ngay  egg_id side                      image_path label  length_mm  \
0    1     0       1    A   dot_1_ngay_0_egg_1_side_A.jpg     T        NaN   
1    1     0       1    B   dot_1_ngay_0_egg_1_side_B.jpg     T        NaN   
2    1     0       1    C   dot_1_ngay_0_egg_1_side_C.jpg     T        NaN   
3    1     0      10    A  dot_1_ngay_0_egg_10_side_A.jpg     T        NaN   
4    1     0      10    B  dot_1_ngay_0_egg_10_side_B.jpg     T        NaN   

   width_mm                                      original_path  
0       NaN                     dot_1_ngay_0\basler_gige_1.jpg  
1       NaN  dot_1_ngay_0\basler_usb_w_38.32_h_49.047_g_1_1...  
2       NaN  dot_1_ngay_0\basler_usb_w_39.728_h_51.387_g_1_...  
3       NaN                    dot_1_ngay_0\basler_gige_10.jpg  
4       NaN  dot_1_ngay_0\basler_usb_w_37.104_h_49.086_g_1_...  


In [7]:
# OPTIMIZED SEGMENTATION WITH PRE-RESIZING
import os
import cv2
import numpy as np
import torch
import gc
from pathlib import Path
from tqdm.auto import tqdm
from segment_anything import sam_model_registry, SamPredictor

# 1. SETUP & LOAD MODEL
SAM_CHECKPOINT_PATH = "/content/sam_vit_h_4b8939.pth"
MODEL_TYPE = "vit_h"
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"[*] Initializing SAM {MODEL_TYPE.upper()} on {DEVICE}...")
sam = sam_model_registry[MODEL_TYPE](checkpoint=SAM_CHECKPOINT_PATH)
sam.to(device=DEVICE)
predictor = SamPredictor(sam)
print("[+] Predictor loaded successfully. Anti-BSOD mode enabled.")

# 2. HELPER FUNCTION: RESIZE HUGE IMAGES
def resize_image_for_sam(image_bgr, max_size=1024):
    """
    Resizes the image if its longest side exceeds max_size.
    This prevents RAM/VRAM explosion and BSOD caused by massive raw photos.
    """
    h, w = image_bgr.shape[:2]
    if max(h, w) > max_size:
        scale = max_size / max(h, w)
        new_w, new_h = int(w * scale), int(h * scale)
        # INTER_AREA is best for shrinking images
        return cv2.resize(image_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return image_bgr

# 3. OPTIMIZED CORE FUNCTION (RGB + MASK)
def predict_and_crop(img_path, rgb_save_path, mask_save_path, padding=5):
    """Uses Center-Point Prompting + outputs BOTH RGB and mask."""

    image_bgr = cv2.imread(str(img_path))
    if image_bgr is None:
        return False

    # --- Resize (anti-BSOD safeguard) ---
    image_bgr = resize_image_for_sam(image_bgr, max_size=1024)

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    h, w, _ = image_rgb.shape

    predictor.set_image(image_rgb)

    # --- Center prompt ---
    center_point = np.array([[w // 2, h // 2]])
    input_label = np.array([1])

    masks, _, _ = predictor.predict(
        point_coords=center_point,
        point_labels=input_label,
        multimask_output=False
    )

    egg_mask = masks[0]

    # Convert to proper binary mask (0 / 255)
    mask_uint8 = (egg_mask > 0).astype(np.uint8) * 255

    y_indices, x_indices = np.where(egg_mask)
    if len(y_indices) == 0 or len(x_indices) == 0:
        return False

    # --- Bounding box ---
    y_min = max(0, y_indices.min() - padding)
    y_max = min(h, y_indices.max() + padding)
    x_min = max(0, x_indices.min() - padding)
    x_max = min(w, x_indices.max() + padding)

    # --- Segmented image ---
    segmented_bgr = np.zeros_like(image_bgr)
    segmented_bgr[egg_mask] = image_bgr[egg_mask]

    cropped_rgb  = segmented_bgr[y_min:y_max, x_min:x_max]
    cropped_mask = mask_uint8[y_min:y_max, x_min:x_max]

    # --- SAVE BOTH ---
    cv2.imwrite(str(rgb_save_path), cropped_rgb)
    cv2.imwrite(str(mask_save_path), cropped_mask)

    return True

[*] Initializing SAM VIT_H on cuda:0...
[+] Predictor loaded successfully. Anti-BSOD mode enabled.


In [8]:
# ==========================================
# PART 1: BATCH SEGMENTATION (RGB + MASK)
# ==========================================

# --- CONFIG (UPDATED FOR DATN) ---
INPUT_DIR = "/content/images"
RGB_DIR   = "/content/drive/MyDrive/DATN/seg/renamed_images"
MASK_DIR  = "/content/drive/MyDrive/DATN/mask/renamed_images"

# --- CREATE OUTPUT DIRS ---
os.makedirs(RGB_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)

# --- LOAD FILES ---
image_files = [
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith((".jpg", ".png"))
]

pbar = tqdm(image_files, desc="Segmenting Part 2", unit="img")

# --- STATS ---
success, failed, skipped = 0, 0, 0

# --- PROCESS LOOP ---
for img_name in pbar:
    in_path   = Path(INPUT_DIR) / img_name
    rgb_path  = Path(RGB_DIR) / img_name
    mask_path = Path(MASK_DIR) / img_name

    # Skip if BOTH outputs already exist
    if rgb_path.exists() and mask_path.exists():
        skipped += 1
        continue

    try:
        ok = predict_and_crop(
            img_path=in_path,
            rgb_save_path=rgb_path,
            mask_save_path=mask_path
        )

        if ok:
            success += 1
        else:
            failed += 1

    except Exception as e:
        print(f"\n❌ Error processing {img_name}: {e}")
        failed += 1

    # Update progress bar
    pbar.set_postfix(
        Success=success,
        Skip=skipped,
        Fail=failed
    )

    # Clear SAM state (important for VRAM stability)
    predictor.reset_image()

# --- FINAL REPORT ---
print("\n✅ COMPLETE!")
print(f"✔ Success: {success}")
print(f"⏭ Skipped: {skipped}")
print(f"❌ Failed: {failed}")

Segmenting Part 2:   0%|          | 0/4633 [00:00<?, ?img/s]


✅ COMPLETE!
✔ Success: 4632
⏭ Skipped: 0
❌ Failed: 1


In [9]:
import os

RGB_DIR   = "/content/drive/MyDrive/DATN/seg/renamed_images"
MASK_DIR  = "/content/drive/MyDrive/DATN/mask/renamed_images"

RGB_ZIP   = "/content/drive/MyDrive/DATN/rgb_images.zip"
MASK_ZIP  = "/content/drive/MyDrive/DATN/mask_images.zip"

print("🗜️ Starting flat compression (top-level only)...")

# --- RGB ---
if os.path.exists(RGB_ZIP):
    print(f"⏭️ Skipping RGB (already exists): {RGB_ZIP}")
elif os.path.exists(RGB_DIR):
    !find {RGB_DIR} -maxdepth 1 -type f -print0 | xargs -0 zip -j -q {RGB_ZIP}
    print(f"✅ RGB images zipped (flat, top-level only) to: {RGB_ZIP}")
else:
    print(f"❌ RGB_DIR not found: {RGB_DIR}")

# --- MASK ---
if os.path.exists(MASK_ZIP):
    print(f"⏭️ Skipping MASK (already exists): {MASK_ZIP}")
elif os.path.exists(MASK_DIR):
    !find {MASK_DIR} -maxdepth 1 -type f -print0 | xargs -0 zip -j -q {MASK_ZIP}
    print(f"✅ Mask images zipped (flat, top-level only) to: {MASK_ZIP}")
else:
    print(f"❌ MASK_DIR not found: {MASK_DIR}")

print("\n✨ All tasks complete!")

🗜️ Starting flat compression (top-level only)...
✅ RGB images zipped (flat, top-level only) to: /content/drive/MyDrive/DATN/rgb_images.zip
✅ Mask images zipped (flat, top-level only) to: /content/drive/MyDrive/DATN/mask_images.zip

✨ All tasks complete!
